In [ ]:
!pip -q install tensorflow-datasets scikit-learn seaborn importlib_resources

import os, json, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 2

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
import importlib
importlib.invalidate_caches()

# The dataset is downloaded by TensorFlow Datasets on first run.
(ds_all, ds_info) = tfds.load(
    "plant_village",
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=True
)

class_names = ds_info.features["label"].names
num_classes = len(class_names)
total_count = int(ds_info.splits["train"].num_examples)

print("Total images:", total_count)
print("Number of classes:", num_classes)
print("\nClass labels:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

In [ ]:
# Preview a few dataset images and labels
plt.figure(figsize=(12, 8))
for i, (image, label) in enumerate(ds_all.take(12)):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(image)
    plt.title(class_names[int(label)])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
train_count = int(total_count * 0.70)
val_count = int(total_count * 0.15)
test_count = total_count - train_count - val_count

shuffled = ds_all.shuffle(
    buffer_size=min(total_count, 10000),
    seed=SEED,
    reshuffle_each_iteration=False
)
train_raw = shuffled.take(train_count)
remaining = shuffled.skip(train_count)
val_raw = remaining.take(val_count)
test_raw = remaining.skip(val_count)

print(f"Train: {train_count}")
print(f"Validation: {val_count}")
print(f"Test: {test_count}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.cast(label, tf.int32)
    return image, label

train_ds = (train_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds = (val_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_ds = (test_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE).prefetch(AUTOTUNE))

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
import tensorflow as tf

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_plant_leaf_cnn.keras", monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("training_accuracy.png", dpi=160)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=160)
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(num_classes)),
    target_names=class_names,
    zero_division=0
))

report_dict = classification_report(
    y_true, y_pred,
    labels=list(range(num_classes)),
    target_names=class_names,
    zero_division=0,
    output_dict=True
)
with open("classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(16, 14))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix — Test Set")
plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=90, fontsize=7)
plt.yticks(ticks, class_names, fontsize=7)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=180)
plt.show()

model.save("plant_leaf_disease_cnn.keras")
print("Saved model: plant_leaf_disease_cnn.keras")

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
if not uploaded:
    print("No file uploaded.")
else:
    image_path = next(iter(uploaded.keys()))
    pil_image = Image.open(image_path).convert("RGB")
    display(pil_image)

    arr = np.array(pil_image.resize(IMG_SIZE)).astype("float32") / 255.0
    probs = model.predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:3]

    print("Top predictions:")
    for idx in top_indices:
        print(f"{class_names[idx]}: {probs[idx]*100:.2f}%")

    best_idx = int(top_indices[0])
    plt.figure(figsize=(6, 4))
    plt.imshow(pil_image)
    plt.axis("off")
    plt.title(f"Predicted: {class_names[best_idx]} ({probs[best_idx]*100:.1f}%)")
    plt.tight_layout()
    plt.savefig("single_image_prediction.png", dpi=160)
    plt.show()

In [ ]:
from google.colab import files
for filename in [
    "training_accuracy.png",
    "training_loss.png",
    "confusion_matrix.png",
    "classification_report.json",
    "plant_leaf_disease_cnn.keras"
]:
    if os.path.exists(filename):
        print("Ready:", filename)
# Uncomment individual lines to download a file:
# files.download("training_accuracy.png")
# files.download("training_loss.png")
# files.download("confusion_matrix.png")
# files.download("classification_report.json")
# files.download("plant_leaf_disease_cnn.keras")